Post-processing of CFD solution
- by py vista library

In [1]:
from scipy.io import mmwrite, mmread
import pyvista as pv
import numpy as np
import pandas as pd
from pyau3d.utils import PltFileUtils, UnkFileUtils
from pyau3d.pv.loader.au3d import arrays2vtk

In [2]:
def specific_energy(rho, p, ux, uy, uz, gamma=1.4):
    return p / ((gamma - 1.0) * rho) + 0.5 * (ux**2 + uy**2 + uz**2)

def conservative_variables(rst, GAMMA):
    """Return conservative variables U on the surface."""
    E = specific_energy(rst.rho, rst.p, rst.ux, rst.uy, rst.uz, GAMMA)

    U = np.column_stack((
        rst.rho,
        rst.rho * rst.ux,
        rst.rho * rst.uy,
        rst.rho * rst.uz,
        rst.rho * E
    ))
    return U

In [21]:
# 1. Read neccessary files
Mesh = 199560
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver
mesh_ver = 3
case_name = "cylinder"

# dir = f"C:/Users/User/Git/flux_jacobian/cases/v{mesh_ver}_mesh/cylinder_{Mesh}_Re{Re}_M{Mach}"
dir = f"/home/ahf25/CFD_2d_cylinder_all/Steady/Ma{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{Re}" 
# dir = f"/home/ahf25/OAT15/OAT15_M0.73_A35" 

# dir = "C:/Users/User/Git/flux_jacobian/cases/OAT15/OAT15_M0.73_A35"
pltfile = PltFileUtils(f"{dir}/{case_name}.plt")
rstfile = UnkFileUtils(f"{dir}/{case_name}.rst", extend=False)  # both rst and unk are fine

fortfile = pd.read_csv(f"{dir}/fort.864", sep=r'\s+', header=None).to_numpy()

rstfile._primitive()
U_list = conservative_variables(rstfile, gamma)
coord  = pltfile.coord

In [22]:
# transform PLT to VTK
mesh = arrays2vtk(pltfile)

# extract the grid from the pv object
domain = pv.wrap(mesh.GetBlock(0))    # "Domain 1"  →  pv.MultiBlock
block  = pv.wrap(domain.GetBlock(0))  # "Volume"    →  pv.UnstructuredGrid

# overlay U_list on the VTK file 
names = ['rho', 'rhou', 'rhov', 'rhow', 'rhoE']

for i in range(5):
    block.point_data[names[i]] = U_list[:,i]

calculate CL on the cylinder wall

In [5]:
def area_normals(coord, ifac3=None, ifac4=None):
    """
    Compute area-weighted normal vectors (Ax, Ay, Az) at each nodes,
    accumulated from triangle and/or quadrilateral faces. No PolyData is
    built; only the (N, 3) array is returned.
 
    For each face, the area-normal vector is:
        triangle : 0.5 * cross(v1 - v0, v2 - v0)
        quad     : sum of the two triangle area-normals from splitting
                   the quad (0,1,2) + (0,2,3)
    Each face's area-normal is split equally among its vertices and summed.

    That is how it accounted from the node-centered formulation
 
    Args:
        coord : (N, 3) array of mesh point coordinates
        ifac3 : (M3, 3) array of triangle connectivity, zero-based (or None)
        ifac4 : (M4, 4) array of quad connectivity, zero-based (or None)
 
    Returns:
        anor : (N, 3) array of area-weighted normals (Ax, Ay, Az) per point
    """
    coord = np.asarray(coord, dtype=float)
    anor = np.zeros_like(coord)
 
    if ifac3 is not None and len(ifac3) > 0:
        ifac3 = np.asarray(ifac3)
        v0, v1, v2 = coord[ifac3[:, 0]], coord[ifac3[:, 1]], coord[ifac3[:, 2]]
        face_anor = 0.5 * np.cross(v1 - v0, v2 - v0)   # (M3, 3)
        contrib = -face_anor / 3.0                      # sign matches ref code
        for k in range(3):
            np.add.at(anor, ifac3[:, k], contrib)
 
    if ifac4 is not None and len(ifac4) > 0:
        ifac4 = np.asarray(ifac4)
        v0, v1, v2, v3 = (coord[ifac4[:, 0]], coord[ifac4[:, 1]],
                          coord[ifac4[:, 2]], coord[ifac4[:, 3]])
        n1 = 0.5 * np.cross(v1 - v0, v2 - v0)
        n2 = 0.5 * np.cross(v2 - v0, v3 - v0)
        face_anor = n1 + n2                              # (M4, 3)
        contrib = -face_anor / 4.0
        for k in range(4):
            np.add.at(anor, ifac4[:, k], contrib)
 
    return anor

def compute_lift_coefficient(flag, direction, U_inf, rho_inf, surface_area,
                              pltfile, rstfile, coord):
    """
    Compute the lift coefficient on a surface identified by `flag`,
    projected along `direction`.

    Parameters
    ----------
    flag : int
        Surface flag identifying which boundary/group to extract (e.g. 7 = cylinder surface).
    direction : array-like, shape (3,)
        Prescribed direction vector to project the force onto (e.g. [0, 1, 0] for lift in y).
    U_inf : float
        Farfield velocity (m/s).
    rho_inf : float
        Farfield density (kg/m^3).
    surface_area : float
        Reference surface area (m^2).
    pltfile : PltFileUtils
        Mesh/geometry file object with `extract_surface_real` method.
    rstfile : object
        Result file object with `.p` (static pressure) array.
    coord : np.ndarray
        Global nodal coordinates array (0-based indexing).

    Returns
    -------
    CL : float
        Lift coefficient.
    """
    # extract nodes, ifac3 and ifac4 on the specified surface
    global_surface_nodes, ifac3, ifac4 = pltfile.extract_surface_real(flag=flag)

    # get surface coordinates (0-based indexing)
    global_surface_coord = coord[global_surface_nodes]

    # get area normals
    surface_area_normals = area_normals(global_surface_coord, ifac3, ifac4)

    # extract static pressure on the surface
    surface_pressure = rstfile.p[global_surface_nodes]

    # project force onto prescribed direction
    force = np.sum(surface_pressure * np.dot(surface_area_normals, direction), axis=0)

    # nondimensionalize
    CL = force / (0.5 * rho_inf * U_inf**2 * surface_area)

    return CL


In [7]:
# define reference parameters
U_inf = 34.02626486 * 2   # farfield velocity (m/s)
rho_inf = 1.60E-05        # farfield density (kg/m^3)
surface_area = 1          # reference surface area m^2

flag = 4
v = [0, 1, 0]  # prescribed direction (lift direction)

CL = compute_lift_coefficient(flag, v, U_inf, rho_inf, surface_area,
                               pltfile, rstfile, coord)
print(f"Lift Coefficent: {CL:.7f}")


Lift Coefficent: -35258.8976745


In [23]:
# plot mesh

pl = pv.Plotter()
pl.add_mesh(block, style='wireframe', color='grey', line_width=0.5)
pl.view_xy()
pl.show()
pl.camera.zoom(1.5) # >1 zooms in, <1 zooms out — tune to taste
pl.save_graphic(f"Re{Re}_M{Mach}_{Mesh}.svg")

Widget(value='<iframe src="http://localhost:42671/index.html?ui=P_0x788b94382a20_10&reconnect=auto" class="pyv…

In [6]:
# ── Plot multiple field  ───────────────────────────────────────────────────────────

# for i in range(5):
#     pl = pv.Plotter(notebook = True)
#     pl.add_mesh(block, scalars=names[i], cmap='RdBu', show_edges=False, edge_color='grey', opacity=0.99)
#     # pl.add_scalar_bar(title='dF_rho / d(rho*u)')
#     pl.view_xy()
#     pl.camera.zoom(50) # >1 zooms in, <1 zooms out — tune to taste
#     pl.camera.SetFocalPoint(0.1, 0, 0)   # shift focus in +x (adjust value to taste)
#     pl.show()
#     # pl.save_graphic(f"Re{Re}_M{Mach}_{names[i]}.svg")
#     pl.save_graphic(f"OAT_M{Mach}_{names[i]}.svg")


# ── Plot one field  ───────────────────────────────────────────────────────────
scalar = "rhou"

def print_pick(point):
    # finds nearest point and prints its value
    pid = block.find_closest_point(point)
    err = block.point_data['rho'][pid]
    xy  = block.points[pid, :2]
    print(f"Node {pid}  x={xy[0]:.4f}  y={xy[1]:.4f}  err={err:.4e}")

pl = pv.Plotter(notebook = False)
pl.add_mesh(block, scalars=scalar, cmap='RdBu', show_edges=False, edge_color='grey', opacity=0.9, pickable = True)
pl.enable_point_picking(callback=print_pick, show_message=True,
                        font_size=10, color='black', point_size=10)
# pl.add_scalar_bar(title='dF_rho / d(rho*u)')

pl.view_xy()
# pl.show()
pl.save_graphic(f"Re{Re}_M{Mach}_{Mesh}_{scalar}.svg")